In [1]:
%pip install --pre azure-ai-projects azure-identity openai azure-search-documents

In [17]:
import os
import json
import time
from datetime import datetime
from pprint import pprint
from pathlib import Path
from typing import Any, List, Dict, Iterable, Union
from packaging.version import Version

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import DatasetVersion
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)

# Azure Search AI

from azure.search.documents import SearchClient
from azure.search.documents.models import (
    QueryType,
    QueryCaptionType,
    QueryAnswerType,
    VectorizableTextQuery,
)
from collections.abc import Mapping
from azure.core.exceptions import ResourceNotFoundError

# To configure

In [23]:
testset_name = "EvaluationAgent_testset_min_wcontext"#"EvaluationAgent_testset_wcontext"
evaluation_id = None#'eval_0f07a91c1d8c4217ae32bc98284e7200'

AGENT_NAME = "EvaluationAgent"
AGENT_VERSION = "2"
model_system_prompt = """
You are a helpful assistant that answer user's question, leveraging at best the context provided.
Be as more concise as possible.
If you do not have a clear evidence, in the context, of the information to be provided, ask user for more information
"""

isAgent = True

if not isAgent:
    model_to_test = "gpt-5"
    model_temperature_to_test = 1
    model_topp_to_test = 1
    model_system_prompt = "You are a helpful assistant that answer with the shortest sentence as possible"

create_new_testset = False #

index_name = "rag-data"
top_k = 3
selected_fields = ["chunk_id", "parent_id", "chunk", "title"]

In [ ]:
AZURE_AI_PROJECT_ENDPOINT = "https://angandin-foundryproject-resource.services.ai.azure.com/api/projects/angandin-foundryproject" # The Azure AI Project project endpoint, as found in the Home page of your Microsoft Foundry portal.
AZURE_AI_MODEL_DEPLOYMENT_NAME = "gpt-4.1" # The deployment name of the AI model, as found under the "Build" page in the "Models" tab in your Foundry project.
DATASET_NAME = testset_name

AZURE_TENANT_ID = "3268e587-3ee5-42d0-b9ac-23fcaf66f510"
AZURE_CLIENT_ID = "ce7e72a5-4ed1-40ac-ab78-fbce8f09bd5e"
AZURE_CLIENT_SECRET = \"<REDACTED-SET-VIA-ENV-OR-KEYVAULT>\"

# Azure Search AI
SEARCH_ENDPOINT   = "https://angandin-eu-aisearch.search.windows.net"  # e.g., https://contoso-search.search.windows.net

In [25]:
# Azure AI Project endpoint
# Example: https://<account_name>.services.ai.azure.com/api/projects/<project_name>
endpoint = AZURE_AI_PROJECT_ENDPOINT
search_endpoint = SEARCH_ENDPOINT

# Model deployment name
# Example: gpt-4o-mini
model_deployment_name = AZURE_AI_MODEL_DEPLOYMENT_NAME

# Dataset details
dataset_name = DATASET_NAME

tenant_id = AZURE_TENANT_ID
client_id = AZURE_CLIENT_ID
client_secret = AZURE_CLIENT_SECRET

In [26]:
# using service principal
cred = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
project_client = AIProjectClient(endpoint=endpoint, credential=cred)

# or using user authentication
# project_client = AIProjectClient( 
#     endpoint=endpoint, 
#     credential=DefaultAzureCredential(), 
# )

search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=cred
    )

### Retrieve from Search AI

In [27]:
APPEND_PRETTY_JSON_IN_QUERY = False
# ======================

def searchai_get_context(search_text, top_k, selected_fields):
    vector_queries = [
        VectorizableTextQuery(
            text="*",
            fields="text_vector",
        )
    ]

    results = search_client.search(
        search_text=search_text,
        top=top_k,
        select=selected_fields,
        include_total_count=True,            # "count": true
        vector_queries=vector_queries,       # "vectorQueries": [...]
        query_type=QueryType.SEMANTIC,       # "queryType": "semantic"
        semantic_configuration_name="rag-data-semantic-configuration",  # "semanticConfiguration"
        query_language="en-us",                # "queryLanguage": "en-us"
        query_rewrites="generative",           # "queryRewrites": "generative"
    )

    results_list = list(results)

    docs = []
    for doc in results_list:
        d = dict(doc) 
        item = {field: d.get(field) for field in selected_fields}
        docs.append(item)


    json_str = json.dumps(docs, ensure_ascii=False, indent=2)
    # print(json_str)
    return json_str

def _to_dict_like(obj: Any) -> Dict:
    """
    Best-effort conversion to a JSON-serializable dict.
    Handles:
      - dict
      - Mapping
      - objects that support 'dict(obj)' (e.g., Azure SearchDocument)
      - generic Python objects with __dict__
    """
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, Mapping):
        return dict(obj)
    # Some SDK types support dict(obj)
    try:
        return dict(obj)  # e.g., azure.search.documents.models.SearchDocument
    except Exception:
        pass
    # Fallback to __dict__ if available
    if hasattr(obj, "__dict__"):
        return vars(obj)
    # Last resort: raise
    raise TypeError(f"Cannot convert object of type '{type(obj).__name__}' to dict")


def ensure_context_array(context_source: Any) -> List[Dict]:
    """
    Normalize any search result into list[dict] as required by your evaluator.
    Accepts list/iterable, dict/single object.
    """
    # If already a string, try to parse JSON (defensive)
    if isinstance(context_source, str):
        try:
            parsed = json.loads(context_source)
            context_source = parsed
        except Exception:
            # Treat string as a single field object
            return [{"text": context_source}]

    # Single dict/object
    if isinstance(context_source, Mapping) or not isinstance(context_source, Iterable) or isinstance(context_source, (bytes, bytearray)):
        return [_to_dict_like(context_source)]  # wrap in list

    # Iterable -> list of dicts
    normalized: List[Dict] = []
    for i, item in enumerate(context_source):
        if isinstance(item, (str, bytes, bytearray)):
            # Try parse JSON string
            try:
                parsed = json.loads(item)
                if isinstance(parsed, Mapping):
                    normalized.append(dict(parsed))
                else:
                    raise ValueError(f"context[{i}] string is not a JSON object.")
            except Exception:
                # As a fallback, wrap plain strings
                normalized.append({"text": item})
        else:
            normalized.append(_to_dict_like(item))
    return normalized


def project_fields(items: List[Dict], fields: List[str]) -> List[Dict]:
    """
    Project each dict to a subset of fields. If fields is empty, returns items unchanged.
    """
    if not fields:
        return items
    out: List[Dict] = []
    for obj in items:
        out.append({f: obj.get(f) for f in fields})
    return out


def json_for_query(value: Any, pretty: bool) -> str:
    """
    Convert a Python object (e.g., list[dict]) to JSON text for concatenating into 'query'.
    """
    if pretty:
        return json.dumps(value, ensure_ascii=False, indent=2)
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def process_file(
    input_path: Path,
    output_path: Path,
    selected_fields: List[str],
    pretty_json_in_query: bool = False,
) -> None:
    """
    - Read the input JSONL file line by line.
    - For each object:
        * call search_ai_context(query)
        * normalize results -> list[dict]
        * (optionally) project fields
        * set obj['context'] = normalized
        * append JSON text of 'context' to obj['query']
    - Write all results at once at the end to output_path.
    """
    processed_lines: List[str] = []

    with input_path.open("r", encoding="utf-8") as fin:
        for line_no, line in enumerate(fin, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Line {line_no} is not valid JSON: {e.msg}") from e

            query = obj.get("query", "")
            if not isinstance(query, str):
                raise ValueError(f"Line {line_no}: 'query' must be a string.")

            # === Call your search ===
            try:
                raw_context = searchai_get_context(query, top_k, selected_fields)
            except Exception as ex:
                # If you prefer to skip failing lines, you can log and continue instead.
                raise RuntimeError(f"Line {line_no}: search_ai_context failed: {ex}") from ex

            # Normalize -> list[dict]
            context_full = ensure_context_array(raw_context)
            # Optional projection
            context_projected = project_fields(context_full, selected_fields)

            # # Assign 'context'
            obj["context"] = context_projected
            obj["system_prompt"] = model_system_prompt

            # Append textual JSON of context to 'query'
            context_text = json_for_query(context_projected, pretty=pretty_json_in_query)

            # obj["context"] = context_text

            original_query = query
            obj["query"] = f"{original_query}\n###\nContext: {context_text}\n###"

            processed_lines.append(json.dumps(obj, ensure_ascii=False))

    # === Write once at the end ===
    with output_path.open("w", encoding="utf-8") as fout:
        fout.write("\n".join(processed_lines) + "\n")

    print(f"✅ Output written to: {output_path.resolve()}")
    return output_path

if create_new_testset:

    testset_wcontext_name = testset_name + "_wcontext"
    testset_input_jsonl  = Path("/lakehouse/default/Files/EvaluationAgent/" + testset_name + ".jsonl")
    testset_output_jsonl = Path("/lakehouse/default/Files/EvaluationAgent/" + testset_wcontext_name + ".jsonl")

    dataset_name = testset_wcontext_name

    output_file_path = process_file(
        input_path=testset_input_jsonl,
        output_path=testset_output_jsonl,
        selected_fields=selected_fields,
        pretty_json_in_query=APPEND_PRETTY_JSON_IN_QUERY,
    )

    testset_version = None
    all_existing_versions_iter = project_client.datasets.list_versions(dataset_name)
    all_existing_versions = list(all_existing_versions_iter)

    if not all_existing_versions:
        testset_version = "1"
    else:
        latest = max(all_existing_versions, key=lambda d: Version(d.version))
        testset_version = str(int(latest.version) + 1)

    try:
        data = project_client.datasets.upload_file(
            name=dataset_name,
            version=testset_version,
            file_path=output_file_path,
        )
    except Exception as e:
        print(f'Error: {e}')

# Run Evaluation

In [28]:
with ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret) as credential:
    with AIProjectClient(endpoint=endpoint, credential=credential) as project_client:
        print("Retrieving existing dataset")

        testset_version = None
        all_existing_versions_iter = project_client.datasets.list_versions(dataset_name)
        all_existing_versions = list(all_existing_versions_iter)

        if not all_existing_versions:
            raise Exception("Testset not found")
        else:
            dataset = max(all_existing_versions, key=lambda d: Version(d.version))
        #dataset = project_client.datasets.get(dataset_name, dataset_version)

        print("Creating an OpenAI client from the AI Project client")
        client = project_client.get_openai_client()

        data_source_config = {
                "type": "custom",
                "item_schema": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string"},
                        "response": {"type": "string"},
                        "context": {"type": "string"},
                        "ground_truth": {"type": "string"},
                        "system_prompt": {"type": "string"}
                    },
                    "required": [],
                },
                "include_sample_schema": True
            }
            
        testing_criteria = [
            {
                "type": "azure_ai_evaluator",
                "name": "violence",
                "evaluator_name": "builtin.violence",
                "data_mapping": {
                    "query": "{{item.query}}",
                    "response": "{{sample.output_text}}",
                },
                "initialization_parameters": {
                    "deployment_name": f"{model_deployment_name}"
                },
            },
            {
                "type": "azure_ai_evaluator",
                "name": "coherence",
                "evaluator_name": "builtin.coherence",
                "initialization_parameters": {
                    "deployment_name": f"{model_deployment_name}",
                    "threshold": 3
                },
                "data_mapping": {
                    "query": "{{item.query}}",
                    "response": "{{sample.output_text}}",
                    "ground_truth": "{{item.ground_truth}}",
                }
            },
            {
                "type": "azure_ai_evaluator",
                "name": "Similarity",
                "evaluator_name": "builtin.similarity",
                "initialization_parameters": {
                    "deployment_name": f"{model_deployment_name}",
                    "threshold": 3
                },
                "data_mapping": {
                    "query": "{{item.query}}",
                    "response": "{{sample.output_text}}",
                    "ground_truth": "{{item.ground_truth}}",
                }
            },
            {
                "type": "azure_ai_evaluator",
                "name": "fluency",
                "evaluator_name": "builtin.fluency",
                "initialization_parameters": {
                    "deployment_name": f"{model_deployment_name}",
                    "threshold": 3
                },
                "data_mapping": {
                    "response": "{{sample.output_text}}"
                },
            },
            {
                "type": "azure_ai_evaluator",
                "name": "relevance",
                "evaluator_name": "builtin.relevance",
                "initialization_parameters": {
                    "deployment_name": f"{model_deployment_name}",
                    "threshold": 3
                },
                "data_mapping": {
                    "query": "{{item.query}}",
                    "response": "{{sample.output_text}}"
                }
            },
            {
                "type": "azure_ai_evaluator",
                "name": "response_completeness",
                "evaluator_name": "builtin.response_completeness",
                "initialization_parameters": {
                    "deployment_name": f"{model_deployment_name}",
                    "threshold": 3
                },
                "data_mapping": {
                    "response": "{{sample.output_text}}",
                    "ground_truth": "{{item.ground_truth}}"
                }
            },
            {
                "type": "azure_ai_evaluator",
                "name": "groundedness",
                "evaluator_name": "builtin.groundedness",
                "initialization_parameters": {
                    "deployment_name": f"{model_deployment_name}",
                    "threshold": 3
                },
                "data_mapping": {
                    "context": "{{item.context}}",
                    "response": "{{sample.output_text}}",
                }
            },
            {
                "type": "azure_ai_evaluator",
                "name": "Custom-Syntax-Evaluator",
                "evaluator_name": "Custom-Syntax-Evaluator",
                "initialization_parameters": {
                    "deployment_name": f"{model_deployment_name}",
                    "threshold": 3
                },
                "data_mapping": {
                    "query": "{{item.query}}",
                    "response": "{{sample.output_text}}",
                    "ground_truth": "{{item.system_prompt}}"
                }
            }
        ]

        # if Agent, expand criterias
        if isAgent:
            for item in testing_criteria:
                item["data_mapping"]["tool_calls"] = "{{sample.tool_calls}}"
                item["data_mapping"]["tool_definitions"] = "{{sample.tool_definitions}}"

        if not evaluation_id:
            print("Creating Eval Group")
            eval_object = client.evals.create(
                name="EvaluationGroup" + "-" + datetime.utcnow().strftime("%Y%m%d%H%M"),
                data_source_config=data_source_config,
                testing_criteria=testing_criteria,
            )
            print(f"Eval Group created: {eval_object.id}")

            evaluation_id = eval_object.id
        else:
            eval_object = client.evals.retrieve(evaluation_id)

        print("Get Eval Group by Id")
        eval_object_response = client.evals.retrieve(eval_object.id)
        # print("Eval Group Response:")
        # pprint(eval_object_response)

        print("Creating Eval Run with Dataset ID")

        data_source={
                "type": "azure_ai_target_completions",
                "input_messages": {
                    "type": "template",
                    "sample": {
                        "type": "object",
                        "properties": {
                            "output_text": {
                                "type": "string"
                            }
                        }
                    }
                },
                "source": {
                    "type": "file_id",
                    "id": dataset.id
                }
            }

        if isAgent:
            name = AGENT_NAME + "-" + datetime.utcnow().strftime("%Y%m%d%H%M")
            data_source['input_messages']['template'] = [
                             {
                                 "role": "user",
                                 "content": "{{item.query}}",
                                 "type": "message"
                             }
                         ]
            data_source['target'] = {
                            "type": "azure_ai_agent",
                            "name": AGENT_NAME,
                            "version": AGENT_VERSION,
                            "tool_descriptions": []
                        }
        else:
            name = model_to_test + "-" + datetime.utcnow().strftime("%Y%m%d%H%M")
            data_source['input_messages']['template'] = [
                                {
                                    "role": "system",
                                    "content": model_system_prompt,
                                    "type": "message"
                                },
                                {
                                    "role": "user",
                                    "content": "{{item.query}}",
                                    "type": "message"
                                },
                            ]
            data_source['target'] = {
                        "type": "azure_ai_model",
                        "model": model_to_test,
                        "sampling_params": {
                            "temperature": model_temperature_to_test,
                            "top_p": model_topp_to_test
                        }
                    }
        eval_run_object = client.evals.runs.create(
            eval_id=eval_object.id,
            name=name,
            metadata={"team": "eval-exp", "scenario": "dataset-id-v1"},
            data_source=data_source
        )

        print(f"Eval Run created: {eval_run_object.id}")
        # pprint(eval_run_object)

        evaluation_run_id = eval_run_object.id

        print("Get Eval Run by Id")
        eval_run_response = client.evals.runs.retrieve(
            run_id=eval_run_object.id,
            eval_id=eval_object.id,
        )
        # print("Eval Run Response:")
        # pprint(eval_run_response)

        # Poll until the run completes or fails
        while True:
            run = client.evals.runs.retrieve(
                run_id=eval_run_response.id, eval_id=eval_object.id
            )
            if run.status in ("completed", "failed"):
                output_items = list(
                    client.evals.runs.output_items.list(
                        run_id=run.id, eval_id=eval_object.id
                    )
                )
                # pprint(output_items)
                print(f"Eval Run Report URL: {run.report_url}")
                break

            time.sleep(30)
            print("Waiting for eval run to complete...")

# Utils

### Retrieve evaluation run and results

In [100]:
# client = project_client.get_openai_client()

# eval_run_results = client.evals.runs.retrieve(
#             run_id=evaluation_run_id,
#             eval_id=evaluation_id,
#         )

# output_items = list(
#     client.evals.runs.output_items.list(
#         run_id=run.id, eval_id=eval_object.id
#     )
# )
# # pprint(output_items)
# print(f"Eval Run Report URL: {run.report_url}")

In [101]:
# project_client.datasets.delete('EvaluationAgent_testset', '1')

# data = project_client.datasets.upload_file(
#     name="EvaluationAgent_testset_min",
#     version="1",
#     file_path="/lakehouse/default/Files/EvaluationAgent/EvaluationAgent_testset_min.jsonl",
# )

In [13]:
l = project_client.evaluators.list()
for i in l:
    print(l)

In [22]:
l = project_client.evaluators.list_latest_versions()
for i in l:
    print(i)